In [1]:
import re
import pandas as pd 
import string
from pathlib import Path

In [2]:
# Txt cleaning

def clean_text(text):
    if pd.isna(text): # if text empty 
        return '' # return empty
        
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)  # remove extra spaces
    
    return text

In [3]:
# extract guest names if present

def guest_names(text):
    if pd.isna(text):
        return []
    return re.findall(r"\((.*?)\)", text)

In [4]:
# combine transcript rows

def create_full_text(rows):
    return "".join(rows)

In [5]:
# create batches
def create_batches(text, batch_size=300):
    """
    batch_size = number of words
    300 is safe for BERT (max ~512 tokens)
    """

    words = text.split()
    batches = []

    for i in range(0, len(words), batch_size):
        batch = " ".join(words[i:i + batch_size])
        batches.append(batch)

    return batches

In [6]:
# process one ep

def process_episode(csv_path):

    df = pd.read_csv(csv_path)

    # clean text
    df["clean_text"] = df["text"].apply(clean_text)

    # extract guest names
    df["guests"] = df["text"].apply(guest_names)

    # reconstruct full transcript
    full_text = create_full_text(df["clean_text"].tolist())

    # create batches
    batches = create_batches(full_text, batch_size=300)

    return df, full_text, batches

In [7]:
episode_path = Path("generated_episodes/ep3.csv")

df, full_text, batches = process_episode(episode_path)

print("Total words:", len(full_text.split()))
print("Number of batches:", len(batches))

print("\nExample batch:\n")
print(batches[4])

Total words: 7197
Number of batches: 24

Example batch:

auf diesen Satz.Man könnte ihn reduzieren, indem man für bezahlbares Wohnen sorgt.Dafür ist aber die Union nicht bereit.Wir wollen genauso, weil das Thema Lohnabstand kam,der Lohn ist gestiegen seit 2005.Wenn wir bei 15 Euro Mindestlohn ansetzen,dann muss sich niemand beschweren.Lasst uns darüber reden,wie wir mittlere und niedrige Einkommen entlasten.Und da, wo zu viel ist, da rangehen.Über den Lohnabstand reden wir sofort.(Grupp-Kofler) Ich würde gerne darüber sprechen,das Thema Totalverweigerer ist total an der Realität vorbei.Das machen wir sofort.Ich möchte aber zuerst meine Kollegin Anna Mayrin die Runde holen und darauf schauen, was jetzt kommt.Nach der letzten Wahl wurde Hartz IV vom Bürgergeld abgelöst.Jetzt wollen CDU und SPD aus dem Bürgergeldeine neue Grundsicherung machen.Frau Mayr, wissen Sie schon, was sich ändern soll außer dem Namen?(Mayr) Ich weiß nicht, ob es irgendwer weiß.Grundsicherung heißt das Ding im Grun

In [8]:
!pip install -U transformers

In [9]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("chkla/parlbert-topic-german")
model = AutoModelForSequenceClassification.from_pretrained("chkla/parlbert-topic-german")

C:\Users\HP\anaconda3\envs\tf_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from transformers import pipeline

pipeline_classification_topics = pipeline(
    "text-classification",
    model="chkla/parlbert-topic-german",
    truncation=True
)

Device set to use cpu


In [11]:
results = []

for i, batch in enumerate(batches):

    prediction = pipeline_classification_topics(batch)

    results.append({
        "batch": i,
        "topic": prediction[0]["label"],
        "score": prediction[0]["score"]
    })

print(results)

[{'batch': 0, 'topic': 'Government', 'score': 0.9975765347480774}, {'batch': 1, 'topic': 'Government', 'score': 0.9959374666213989}, {'batch': 2, 'topic': 'Social', 'score': 0.9974742531776428}, {'batch': 3, 'topic': 'Social', 'score': 0.8853399753570557}, {'batch': 4, 'topic': 'Social', 'score': 0.9682576656341553}, {'batch': 5, 'topic': 'Social', 'score': 0.998213529586792}, {'batch': 6, 'topic': 'Social', 'score': 0.9344240427017212}, {'batch': 7, 'topic': 'Labor', 'score': 0.7301367521286011}, {'batch': 8, 'topic': 'Labor', 'score': 0.9628084301948547}, {'batch': 9, 'topic': 'Labor', 'score': 0.9990092515945435}, {'batch': 10, 'topic': 'Social', 'score': 0.908444881439209}, {'batch': 11, 'topic': 'Social', 'score': 0.5548296570777893}, {'batch': 12, 'topic': 'Labor', 'score': 0.7451211810112}, {'batch': 13, 'topic': 'Labor', 'score': 0.9927686452865601}, {'batch': 14, 'topic': 'Labor', 'score': 0.9991360306739807}, {'batch': 15, 'topic': 'Labor', 'score': 0.998979389667511}, {'batc